In [1]:
!pip install sentence-transformers scikit-learn pandas numpy imbalanced-learn

In [2]:
# imports
import pandas as pd
import numpy as np
import time
import os

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

print("Imports completed")

Imports completed


In [3]:
# load the data and look at it
df = pd.read_csv("fake_job_postings.csv")
print(df.shape)   # (rows, columns)
df.head()

(17880, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [4]:
# check the class balance
df["fraudulent"].value_counts()

fraudulent
0    17014
1      866
Name: count, dtype: int64

In [5]:
# merge the four text columns into one
text_cols = ["company_profile", "description", "requirements", "benefits"]
df[text_cols] = df[text_cols].fillna("")           # blanks -> empty string
df["combined_text"] = df[text_cols].agg(" ".join, axis=1)

df["combined_text"].iloc[0]   # look at the first merged post

"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly curated information about food and cooking. We attract the most talented home cooks and contributors in the country; we also publish well-known professionals like Mario Batali, Gwyneth Paltrow, and Danny Meyer. And we have partnerships with Whole Foods Market and Random House.Food52 has been named the best food website by the James Beard Foundation and IACP, and has been featured in the New York Times, NPR, Pando Daily, TechCrunch, and on the Today Show.We're located in Chelsea, in New York City. Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub,

In [6]:
# load the Sentence Transformer model
local_dir = os.path.join(os.getcwd(), "all-MiniLM-L6-v2")
if not os.path.exists(local_dir):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="sentence-transformers/all-MiniLM-L6-v2", local_dir=local_dir)

model = SentenceTransformer(local_dir)
print("Model loaded successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully


In [7]:
# creating embeddings for all the job posts (or load cached ones if already saved)
if os.path.exists("embeddings.npy"):
    embeddings = np.load("embeddings.npy")
    print("Loaded cached embeddings.npy, shape:", embeddings.shape)
else:
    start = time.time()
    embeddings = model.encode(df["combined_text"].tolist(), show_progress_bar=True)
    end = time.time()
    print("Time taken:", (end - start) / 60, "minutes")
    print("Shape:", embeddings.shape)
    np.save("embeddings.npy", embeddings)
    print("Saved embeddings.npy")

Loaded cached embeddings.npy, shape: (17880, 384)


In [8]:
# setting up features (xs) and labels (ys)
xs = embeddings
ys = df["fraudulent"].values

# split into training and test sets BEFORE any resampling
# (SMOTE must never see the test set, or evaluation becomes unrealistic)
xs_train, xs_test, ys_train, ys_test = train_test_split(
    xs, ys, test_size=0.20, random_state=42, stratify=ys
)

print("Training set:", xs_train.shape)
print("Test set:", xs_test.shape)
print("Training class balance before SMOTE:", np.bincount(ys_train))

Training set: (14304, 384)
Test set: (3576, 384)
Training class balance before SMOTE: [13611   693]


In [10]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # set to your actual core count

In [11]:
# apply SMOTE to the training set only
# https://imbalanced-learn.org/stable/references/generated/imblearn.over_sampling.SMOTE.html
smote = SMOTE(
    sampling_strategy="auto",  # resample all classes but the majority up to it
    random_state=42,
    k_neighbors=5,
)
xs_train_res, ys_train_res = smote.fit_resample(xs_train, ys_train)

print("Training class balance after SMOTE:", np.bincount(ys_train_res))
print("Resampled training set:", xs_train_res.shape)

Training class balance after SMOTE: [13611 13611]
Resampled training set: (27222, 384)


In [12]:
# training Random Forest classifier on the SMOTE-resampled training data
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(xs_train_res, ys_train_res)

# checking the accuracy on train (resampled) and test (original, untouched) sets
print("Train accuracy (resampled):", accuracy_score(ys_train_res, rf_clf.predict(xs_train_res)))
print("Test accuracy:", accuracy_score(ys_test, rf_clf.predict(xs_test)))

Train accuracy (resampled): 1.0
Test accuracy: 0.9818232662192393


In [13]:
# testing to see how well the classifier catches fake posts (not just overall accuracy)
predictions = rf_clf.predict(xs_test)
print(confusion_matrix(ys_test, predictions))
print(classification_report(ys_test, predictions, target_names=["genuine", "fraudulent"]))

[[3400    3]
 [  62  111]]
              precision    recall  f1-score   support

     genuine       0.98      1.00      0.99      3403
  fraudulent       0.97      0.64      0.77       173

    accuracy                           0.98      3576
   macro avg       0.98      0.82      0.88      3576
weighted avg       0.98      0.98      0.98      3576

